# Portfolio Backtesting Framework

        Advanced Colab-first quantitative finance project for testing reusable portfolio strategies with Yahoo Finance data, transaction costs, professional performance analytics, and interactive Plotly tear sheets.

        This notebook writes a production-style Python package into `src/`, imports it, runs multiple strategies, compares them against a benchmark, and exports research artifacts.


## Colab Cell 01 - Install Dependencies

        Installs the core research stack. `vectorbt` and `backtrader` are optional adapters; the main engine does not depend on them.


In [ ]:

        import subprocess
        import sys


        def install_packages(packages):
            command = [sys.executable, "-m", "pip", "install", "-q", *packages]
            subprocess.check_call(command)


        CORE_PACKAGES = [
            "pandas>=2.2.0",
            "numpy>=1.26.0",
            "yfinance>=0.2.40",
            "plotly>=5.20.0",
            "scipy>=1.11.0",
            "ipywidgets>=8.1.0",
            "nbformat>=5.9.0",
        ]

        OPTIONAL_PACKAGES = [
            "vectorbt>=0.27.0",
            "backtrader>=1.9.78.123",
        ]

        install_packages(CORE_PACKAGES)

        try:
            install_packages(OPTIONAL_PACKAGES)
        except Exception as exc:
            print(f"Optional adapter install failed: {exc}")
            print("The core backtesting engine will still run.")


## Colab Cell 02 - Create Project Structure

        Builds the same folder structure used in the GitHub-ready repository.


In [ ]:

        from pathlib import Path

        PROJECT_ROOT = Path.cwd()
        for path in [
            "src/portfolio_backtester",
            "configs",
            "reports",
            "data/raw",
            "data/processed",
        ]:
            (PROJECT_ROOT / path).mkdir(parents=True, exist_ok=True)

        print(f"Project root: {PROJECT_ROOT}")


## Colab Cell 03 - Requirements File

        Saves a reproducible dependency list for GitHub users.


In [ ]:
%%writefile requirements.txt
pandas>=2.2.0
numpy>=1.26.0
yfinance>=0.2.40
plotly>=5.20.0
scipy>=1.11.0
ipywidgets>=8.1.0
vectorbt>=0.27.0
backtrader>=1.9.78.123
nbformat>=5.9.0


## Colab Cell 04 - Package Init


In [ ]:
%%writefile src/portfolio_backtester/__init__.py
"""Reusable portfolio backtesting framework."""

from .config import BacktestConfig
from .data import MarketData, YahooFinanceDataLoader
from .engine import BacktestEngine, BacktestResult
from .metrics import compute_performance_metrics
from .strategies import (
    BaseStrategy,
    DualMomentumStrategy,
    EqualWeightStrategy,
    VolatilityTargetMomentumStrategy,
)

__all__ = [
    "BacktestConfig",
    "BacktestEngine",
    "BacktestResult",
    "BaseStrategy",
    "DualMomentumStrategy",
    "EqualWeightStrategy",
    "MarketData",
    "VolatilityTargetMomentumStrategy",
    "YahooFinanceDataLoader",
    "compute_performance_metrics",
]


## Colab Cell 05 - Configuration Layer


In [ ]:
%%writefile src/portfolio_backtester/config.py
"""Configuration objects for the portfolio backtesting framework."""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Iterable


def _as_tuple(values: Iterable[str]) -> tuple[str, ...]:
    cleaned = tuple(str(value).strip().upper() for value in values if str(value).strip())
    if not cleaned:
        raise ValueError("At least one ticker is required.")
    return cleaned


@dataclass(frozen=True)
class BacktestConfig:
    """Runtime configuration shared by data, strategy, and execution layers."""

    tickers: tuple[str, ...] = field(default_factory=lambda: ("SPY", "QQQ", "TLT", "GLD"))
    benchmark: str = "SPY"
    start: str = "2015-01-01"
    end: str | None = None
    initial_cash: float = 100_000.0
    commission_bps: float = 1.0
    slippage_bps: float = 2.0
    rebalance_frequency: str = "M"
    risk_free_rate: float = 0.02
    max_gross_leverage: float = 1.0
    allow_short: bool = False
    min_observations: int = 252

    def __post_init__(self) -> None:
        object.__setattr__(self, "tickers", _as_tuple(self.tickers))
        object.__setattr__(self, "benchmark", str(self.benchmark).strip().upper())

        if not self.benchmark:
            raise ValueError("benchmark cannot be empty.")
        if self.initial_cash <= 0:
            raise ValueError("initial_cash must be positive.")
        if self.commission_bps < 0 or self.slippage_bps < 0:
            raise ValueError("Transaction cost assumptions cannot be negative.")
        if self.max_gross_leverage <= 0:
            raise ValueError("max_gross_leverage must be positive.")
        if self.min_observations < 20:
            raise ValueError("min_observations should be at least 20 trading observations.")

    @property
    def all_symbols(self) -> tuple[str, ...]:
        """Return portfolio tickers plus benchmark, preserving order."""

        symbols = list(self.tickers)
        if self.benchmark not in symbols:
            symbols.append(self.benchmark)
        return tuple(symbols)

    @property
    def total_cost_rate(self) -> float:
        """Round-trip trading cost rate used by the execution simulator."""

        return (self.commission_bps + self.slippage_bps) / 10_000.0


## Colab Cell 06 - Data Collection Pipeline


In [ ]:
%%writefile src/portfolio_backtester/data.py
"""Market data loading and cleaning utilities."""

from __future__ import annotations

import time
from dataclasses import dataclass
from typing import Sequence

import numpy as np
import pandas as pd

from .config import BacktestConfig


@dataclass(frozen=True)
class MarketData:
    """Cleaned market data aligned to a single trading calendar."""

    prices: pd.DataFrame
    returns: pd.DataFrame
    volumes: pd.DataFrame
    raw: pd.DataFrame


class YahooFinanceDataLoader:
    """Download and normalize daily OHLCV data from Yahoo Finance via yfinance."""

    def __init__(self, config: BacktestConfig, retries: int = 3, pause_seconds: float = 1.5) -> None:
        self.config = config
        self.retries = max(1, int(retries))
        self.pause_seconds = max(0.0, float(pause_seconds))

    def download(self) -> MarketData:
        """Download data and return adjusted close prices, returns, and volume."""

        try:
            import yfinance as yf
        except ImportError as exc:
            raise RuntimeError(
                "yfinance is required for Yahoo Finance downloads. "
                "Install it with `pip install yfinance`."
            ) from exc

        symbols = self.config.all_symbols
        raw: pd.DataFrame | None = None
        last_error: Exception | None = None

        for attempt in range(1, self.retries + 1):
            try:
                raw = yf.download(
                    list(symbols),
                    start=self.config.start,
                    end=self.config.end,
                    auto_adjust=True,
                    group_by="ticker",
                    progress=False,
                    threads=True,
                    repair=True,
                )
                if raw is not None and not raw.empty:
                    break
            except Exception as exc:  # yfinance can raise transport and parsing errors.
                last_error = exc
            time.sleep(self.pause_seconds * attempt)

        if raw is None or raw.empty:
            raise RuntimeError(f"Yahoo Finance returned no data. Last error: {last_error}")

        prices = self._extract_field(raw, "Close", symbols)
        volumes = self._extract_field(raw, "Volume", symbols)
        prices = self._clean_prices(prices)
        volumes = self._clean_volumes(volumes, prices.index)

        missing = sorted(set(self.config.tickers).difference(prices.columns))
        if missing:
            raise RuntimeError(f"No usable adjusted close prices were found for: {missing}")

        if len(prices) < self.config.min_observations:
            raise RuntimeError(
                f"Only {len(prices)} rows were downloaded. "
                f"Need at least {self.config.min_observations} observations."
            )

        returns = prices.pct_change(fill_method=None).replace([np.inf, -np.inf], np.nan).fillna(0.0)
        return MarketData(prices=prices, returns=returns, volumes=volumes, raw=raw)

    @staticmethod
    def _extract_field(raw: pd.DataFrame, field: str, symbols: Sequence[str]) -> pd.DataFrame:
        """Extract one OHLCV field from either single-index or MultiIndex yfinance output."""

        if isinstance(raw.columns, pd.MultiIndex):
            level_0 = raw.columns.get_level_values(0)
            level_1 = raw.columns.get_level_values(1)
            if field in level_0:
                frame = raw.xs(field, axis=1, level=0, drop_level=True)
            elif field in level_1:
                frame = raw.xs(field, axis=1, level=1, drop_level=True)
            else:
                raise RuntimeError(f"Field `{field}` was not present in downloaded data.")
        else:
            if field not in raw.columns:
                raise RuntimeError(f"Field `{field}` was not present in downloaded data.")
            if len(symbols) != 1:
                raise RuntimeError("Expected MultiIndex columns for multi-asset data.")
            frame = raw[[field]].rename(columns={field: symbols[0]})

        frame = frame.copy()
        frame.columns = [str(col).upper() for col in frame.columns]
        ordered = [symbol for symbol in symbols if symbol in frame.columns]
        return frame.loc[:, ordered]

    @staticmethod
    def _clean_prices(prices: pd.DataFrame) -> pd.DataFrame:
        """Clean adjusted close prices without fabricating a trading history."""

        cleaned = prices.copy()
        cleaned.index = pd.to_datetime(cleaned.index).tz_localize(None)
        cleaned = cleaned.sort_index()
        cleaned = cleaned.replace([np.inf, -np.inf], np.nan)
        cleaned = cleaned.dropna(axis=1, how="all")
        cleaned = cleaned.ffill()
        cleaned = cleaned.dropna(axis=0, how="all")
        return cleaned

    @staticmethod
    def _clean_volumes(volumes: pd.DataFrame, index: pd.Index) -> pd.DataFrame:
        """Clean volume data and align it to the price calendar."""

        cleaned = volumes.copy()
        cleaned.index = pd.to_datetime(cleaned.index).tz_localize(None)
        cleaned = cleaned.sort_index().reindex(index)
        cleaned = cleaned.replace([np.inf, -np.inf], np.nan)
        return cleaned.fillna(0.0)


## Colab Cell 07 - Feature Engineering


In [ ]:
%%writefile src/portfolio_backtester/features.py
"""Feature engineering for cross-asset portfolio strategies."""

from __future__ import annotations

import numpy as np
import pandas as pd


def simple_moving_average(prices: pd.DataFrame, window: int) -> pd.DataFrame:
    """Compute rolling simple moving averages."""

    return prices.rolling(window=window, min_periods=max(5, window // 3)).mean()


def momentum(prices: pd.DataFrame, lookback: int = 126) -> pd.DataFrame:
    """Compute lookback total return momentum."""

    return prices.pct_change(periods=lookback, fill_method=None)


def realized_volatility(returns: pd.DataFrame, lookback: int = 63, periods_per_year: int = 252) -> pd.DataFrame:
    """Compute annualized rolling realized volatility."""

    return returns.rolling(window=lookback, min_periods=max(10, lookback // 3)).std() * np.sqrt(periods_per_year)


def rsi(prices: pd.DataFrame, window: int = 14) -> pd.DataFrame:
    """Compute a vectorized Relative Strength Index."""

    delta = prices.diff()
    gains = delta.clip(lower=0)
    losses = -delta.clip(upper=0)
    avg_gain = gains.ewm(alpha=1 / window, adjust=False, min_periods=window).mean()
    avg_loss = losses.ewm(alpha=1 / window, adjust=False, min_periods=window).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))


def zscore(values: pd.DataFrame, lookback: int = 126) -> pd.DataFrame:
    """Compute rolling z-scores for cross-asset ranking or normalization."""

    rolling_mean = values.rolling(window=lookback, min_periods=max(20, lookback // 3)).mean()
    rolling_std = values.rolling(window=lookback, min_periods=max(20, lookback // 3)).std()
    return (values - rolling_mean) / rolling_std.replace(0, np.nan)


def build_feature_panel(prices: pd.DataFrame, returns: pd.DataFrame) -> dict[str, pd.DataFrame]:
    """Build the default feature dictionary used by example strategies."""

    return {
        "sma_50": simple_moving_average(prices, 50),
        "sma_200": simple_moving_average(prices, 200),
        "momentum_63": momentum(prices, 63),
        "momentum_126": momentum(prices, 126),
        "volatility_63": realized_volatility(returns, 63),
        "rsi_14": rsi(prices, 14),
    }


## Colab Cell 08 - Strategy Library


In [ ]:
%%writefile src/portfolio_backtester/strategies.py
"""Reusable strategy definitions that emit target portfolio weights."""

from __future__ import annotations

from abc import ABC, abstractmethod
from dataclasses import dataclass

import numpy as np
import pandas as pd

from .config import BacktestConfig
from .features import build_feature_panel, momentum, realized_volatility, simple_moving_average


def rebalance_mask(index: pd.Index, frequency: str) -> pd.Series:
    """Return True on the last available observation of each rebalance period."""

    if len(index) == 0:
        return pd.Series(dtype=bool)
    periods = pd.Series(pd.DatetimeIndex(index).to_period(frequency), index=index)
    mask = periods.ne(periods.shift(-1)).fillna(True)
    mask.iloc[0] = True
    return mask


def normalize_long_only(scores: pd.DataFrame, max_gross_leverage: float = 1.0) -> pd.DataFrame:
    """Convert non-negative scores into long-only weights with cash allowed."""

    clipped = scores.clip(lower=0.0).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    row_sums = clipped.sum(axis=1)
    weights = clipped.div(row_sums.replace(0, np.nan), axis=0).fillna(0.0)
    return weights * min(float(max_gross_leverage), 1.0)


class BaseStrategy(ABC):
    """Abstract strategy contract.

    Strategies should use information available at the close and emit target
    weights. The engine shifts weights by one session before execution to avoid
    look-ahead bias.
    """

    name = "base_strategy"

    @abstractmethod
    def generate_weights(
        self,
        prices: pd.DataFrame,
        returns: pd.DataFrame,
        config: BacktestConfig,
        features: dict[str, pd.DataFrame] | None = None,
    ) -> pd.DataFrame:
        """Return a daily DataFrame of target weights indexed like prices."""


@dataclass
class EqualWeightStrategy(BaseStrategy):
    """Periodic equal-weight portfolio with cash for unavailable assets."""

    name: str = "equal_weight"

    def generate_weights(
        self,
        prices: pd.DataFrame,
        returns: pd.DataFrame,
        config: BacktestConfig,
        features: dict[str, pd.DataFrame] | None = None,
    ) -> pd.DataFrame:
        universe = list(config.tickers)
        available = prices[universe].notna().astype(float)
        weights = available.div(available.sum(axis=1).replace(0, np.nan), axis=0).fillna(0.0)
        mask = rebalance_mask(prices.index, config.rebalance_frequency)
        return weights.where(mask, np.nan).ffill().fillna(0.0)


@dataclass
class DualMomentumStrategy(BaseStrategy):
    """Long the strongest assets with positive absolute momentum."""

    lookback: int = 126
    top_n: int = 2
    name: str = "dual_momentum"

    def generate_weights(
        self,
        prices: pd.DataFrame,
        returns: pd.DataFrame,
        config: BacktestConfig,
        features: dict[str, pd.DataFrame] | None = None,
    ) -> pd.DataFrame:
        universe = list(config.tickers)
        scores = momentum(prices[universe], self.lookback)
        positive_scores = scores.where(scores > 0.0, 0.0)
        ranks = positive_scores.rank(axis=1, ascending=False, method="first")
        selected = positive_scores.where(ranks <= self.top_n, 0.0)
        equal_selected = selected.gt(0.0).astype(float)
        weights = normalize_long_only(equal_selected, config.max_gross_leverage)
        mask = rebalance_mask(prices.index, config.rebalance_frequency)
        return weights.where(mask, np.nan).ffill().fillna(0.0)


@dataclass
class VolatilityTargetMomentumStrategy(BaseStrategy):
    """Trend-following cross-asset momentum with inverse-volatility sizing."""

    momentum_lookback: int = 126
    volatility_lookback: int = 63
    trend_window: int = 200
    name: str = "vol_target_momentum"

    def generate_weights(
        self,
        prices: pd.DataFrame,
        returns: pd.DataFrame,
        config: BacktestConfig,
        features: dict[str, pd.DataFrame] | None = None,
    ) -> pd.DataFrame:
        universe = list(config.tickers)
        local_features = features or build_feature_panel(prices, returns)
        mom = local_features.get("momentum_126", momentum(prices[universe], self.momentum_lookback))[universe]
        vol = local_features.get("volatility_63", realized_volatility(returns[universe], self.volatility_lookback))[universe]
        trend = prices[universe] > simple_moving_average(prices[universe], self.trend_window)

        risk_adjusted_score = mom.clip(lower=0.0) / vol.replace(0, np.nan)
        risk_adjusted_score = risk_adjusted_score.where(trend, 0.0)
        weights = normalize_long_only(risk_adjusted_score, config.max_gross_leverage)
        mask = rebalance_mask(prices.index, config.rebalance_frequency)
        return weights.where(mask, np.nan).ffill().fillna(0.0)


## Colab Cell 09 - Performance Metrics


In [ ]:
%%writefile src/portfolio_backtester/metrics.py
"""Portfolio performance analytics."""

from __future__ import annotations

import math

import numpy as np
import pandas as pd


def infer_periods_per_year(index: pd.Index) -> int:
    """Infer an annualization factor from a DatetimeIndex."""

    if len(index) < 3:
        return 252
    dates = pd.DatetimeIndex(index)
    median_days = np.median(np.diff(dates.values).astype("timedelta64[D]").astype(float))
    if median_days <= 2:
        return 252
    if median_days <= 8:
        return 52
    if median_days <= 35:
        return 12
    return 1


def equity_curve(returns: pd.Series, initial_value: float = 1.0) -> pd.Series:
    """Convert periodic returns into an equity curve."""

    clean = returns.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return initial_value * (1.0 + clean).cumprod()


def drawdown_series(equity: pd.Series) -> pd.Series:
    """Compute drawdown as percentage decline from running peak."""

    running_peak = equity.cummax()
    return equity / running_peak - 1.0


def compute_performance_metrics(
    returns: pd.Series,
    benchmark_returns: pd.Series | None = None,
    risk_free_rate: float = 0.0,
) -> pd.Series:
    """Compute institutional portfolio statistics."""

    clean = returns.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    clean = clean.astype(float)
    periods_per_year = infer_periods_per_year(clean.index)
    rf_periodic = (1.0 + risk_free_rate) ** (1.0 / periods_per_year) - 1.0
    excess = clean - rf_periodic
    curve = equity_curve(clean)
    drawdowns = drawdown_series(curve)
    n_periods = max(len(clean), 1)

    ending_value = float(curve.iloc[-1])
    cagr = ending_value ** (periods_per_year / n_periods) - 1.0 if ending_value > 0 else np.nan
    annual_volatility = clean.std(ddof=0) * math.sqrt(periods_per_year)
    downside = clean.where(clean < rf_periodic, 0.0)
    downside_volatility = downside.std(ddof=0) * math.sqrt(periods_per_year)
    max_drawdown = float(drawdowns.min())

    sharpe = _safe_divide(excess.mean() * periods_per_year, excess.std(ddof=0) * math.sqrt(periods_per_year))
    sortino = _safe_divide(excess.mean() * periods_per_year, downside_volatility)
    calmar = _safe_divide(cagr, abs(max_drawdown))

    metrics = {
        "Total Return": ending_value - 1.0,
        "CAGR": cagr,
        "Annual Volatility": annual_volatility,
        "Sharpe Ratio": sharpe,
        "Sortino Ratio": sortino,
        "Calmar Ratio": calmar,
        "Max Drawdown": max_drawdown,
        "Best Period": clean.max(),
        "Worst Period": clean.min(),
        "Hit Rate": (clean > 0).mean(),
        "Skew": clean.skew(),
        "Kurtosis": clean.kurtosis(),
        "VaR 95": clean.quantile(0.05),
        "CVaR 95": clean[clean <= clean.quantile(0.05)].mean(),
    }

    if benchmark_returns is not None:
        benchmark = benchmark_returns.reindex(clean.index).replace([np.inf, -np.inf], np.nan).fillna(0.0)
        active = clean - benchmark
        covariance = np.cov(clean, benchmark, ddof=0)[0, 1] if benchmark.std(ddof=0) > 0 else np.nan
        beta = covariance / np.var(benchmark, ddof=0) if benchmark.var(ddof=0) > 0 else np.nan
        alpha = (clean.mean() - beta * benchmark.mean()) * periods_per_year if pd.notna(beta) else np.nan
        tracking_error = active.std(ddof=0) * math.sqrt(periods_per_year)
        metrics.update(
            {
                "Benchmark CAGR": equity_curve(benchmark).iloc[-1] ** (periods_per_year / n_periods) - 1.0,
                "Alpha": alpha,
                "Beta": beta,
                "Tracking Error": tracking_error,
                "Information Ratio": _safe_divide(active.mean() * periods_per_year, tracking_error),
                "Active Return": clean.sum() - benchmark.sum(),
            }
        )

    return pd.Series(metrics, dtype=float)


def _safe_divide(numerator: float, denominator: float) -> float:
    if denominator is None or not np.isfinite(denominator) or abs(denominator) < 1e-12:
        return np.nan
    return float(numerator / denominator)


## Colab Cell 10 - Backtest Execution Engine


In [ ]:
%%writefile src/portfolio_backtester/engine.py
"""Portfolio execution simulator."""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import pandas as pd

from .config import BacktestConfig
from .metrics import compute_performance_metrics, drawdown_series
from .strategies import BaseStrategy


@dataclass(frozen=True)
class BacktestResult:
    """Container returned by the portfolio engine."""

    strategy_name: str
    config: BacktestConfig
    weights: pd.DataFrame
    target_weights: pd.DataFrame
    portfolio_returns: pd.Series
    equity: pd.Series
    drawdown: pd.Series
    turnover: pd.Series
    costs: pd.Series
    metrics: pd.Series
    benchmark_returns: pd.Series | None = None
    benchmark_equity: pd.Series | None = None


class BacktestEngine:
    """Long-only, cash-aware, transaction-cost-aware portfolio backtester."""

    def __init__(self, config: BacktestConfig) -> None:
        self.config = config

    def run(
        self,
        prices: pd.DataFrame,
        returns: pd.DataFrame,
        strategy: BaseStrategy,
        features: dict[str, pd.DataFrame] | None = None,
    ) -> BacktestResult:
        """Generate strategy weights, simulate execution, and calculate metrics."""

        missing = sorted(set(self.config.tickers).difference(prices.columns))
        if missing:
            raise ValueError(f"Price data is missing configured tickers: {missing}")

        prices = prices.sort_index()
        returns = returns.reindex(prices.index).fillna(0.0)
        target_weights = strategy.generate_weights(prices, returns, self.config, features)
        target_weights = self._sanitize_weights(target_weights.reindex(prices.index).fillna(0.0))

        execution_weights = target_weights.shift(1).fillna(0.0)
        portfolio = self._simulate(returns[list(self.config.tickers)], execution_weights[list(self.config.tickers)])

        benchmark_returns = None
        benchmark_equity = None
        if self.config.benchmark in returns.columns:
            benchmark_returns = returns[self.config.benchmark].reindex(portfolio["returns"].index).fillna(0.0)
            benchmark_equity = self.config.initial_cash * (1.0 + benchmark_returns).cumprod()

        metrics = compute_performance_metrics(
            portfolio["returns"],
            benchmark_returns=benchmark_returns,
            risk_free_rate=self.config.risk_free_rate,
        )

        return BacktestResult(
            strategy_name=strategy.name,
            config=self.config,
            weights=portfolio["weights"],
            target_weights=target_weights,
            portfolio_returns=portfolio["returns"],
            equity=portfolio["equity"],
            drawdown=drawdown_series(portfolio["equity"]),
            turnover=portfolio["turnover"],
            costs=portfolio["costs"],
            metrics=metrics,
            benchmark_returns=benchmark_returns,
            benchmark_equity=benchmark_equity,
        )

    def _simulate(self, returns: pd.DataFrame, execution_weights: pd.DataFrame) -> dict[str, pd.Series | pd.DataFrame]:
        """Simulate close-to-close returns with rebalance-only turnover."""

        index = returns.index
        columns = list(returns.columns)
        returns_array = returns.replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(dtype=float)
        target_array = execution_weights.reindex(index).fillna(0.0).to_numpy(dtype=float)
        target_array = np.nan_to_num(target_array, nan=0.0, posinf=0.0, neginf=0.0)

        n_rows, n_assets = returns_array.shape
        equity = np.empty(n_rows, dtype=float)
        portfolio_returns = np.zeros(n_rows, dtype=float)
        turnover = np.zeros(n_rows, dtype=float)
        costs = np.zeros(n_rows, dtype=float)
        realized_weights = np.zeros((n_rows, n_assets), dtype=float)

        equity[0] = self.config.initial_cash
        current_weights = np.zeros(n_assets, dtype=float)
        previous_target = np.zeros(n_assets, dtype=float)

        for row in range(1, n_rows):
            asset_returns = returns_array[row]
            gross_return = float(np.dot(current_weights, asset_returns))
            pre_cost_equity = equity[row - 1] * (1.0 + gross_return)

            if pre_cost_equity <= 0 or not np.isfinite(pre_cost_equity):
                raise RuntimeError(f"Portfolio equity became invalid on {index[row]}.")

            drifted_value = current_weights * equity[row - 1] * (1.0 + asset_returns)
            drifted_weights = drifted_value / pre_cost_equity
            desired_target = target_array[row]
            should_rebalance = not np.allclose(desired_target, previous_target, atol=1e-10, rtol=1e-8)

            if should_rebalance:
                turnover[row] = float(np.abs(desired_target - drifted_weights).sum())
                costs[row] = pre_cost_equity * turnover[row] * self.config.total_cost_rate
                equity[row] = pre_cost_equity - costs[row]
                current_weights = desired_target.copy()
                previous_target = desired_target.copy()
            else:
                equity[row] = pre_cost_equity
                current_weights = drifted_weights

            portfolio_returns[row] = equity[row] / equity[row - 1] - 1.0
            realized_weights[row] = current_weights

        return {
            "equity": pd.Series(equity, index=index, name="Equity"),
            "returns": pd.Series(portfolio_returns, index=index, name="Portfolio Return"),
            "turnover": pd.Series(turnover, index=index, name="Turnover"),
            "costs": pd.Series(costs, index=index, name="Trading Costs"),
            "weights": pd.DataFrame(realized_weights, index=index, columns=columns),
        }

    def _sanitize_weights(self, weights: pd.DataFrame) -> pd.DataFrame:
        """Enforce portfolio constraints before orders reach the simulator."""

        clean = weights.copy().replace([np.inf, -np.inf], np.nan).fillna(0.0)
        if not self.config.allow_short:
            clean = clean.clip(lower=0.0)

        gross = clean.abs().sum(axis=1)
        too_large = gross > self.config.max_gross_leverage
        if too_large.any():
            clean.loc[too_large] = clean.loc[too_large].div(gross.loc[too_large], axis=0) * self.config.max_gross_leverage
        return clean


## Colab Cell 11 - Professional Visualizations


In [ ]:
%%writefile src/portfolio_backtester/visualization.py
"""Plotly visualizations for portfolio tear sheets."""

from __future__ import annotations

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from .engine import BacktestResult
from .metrics import drawdown_series, equity_curve


CHART_TEMPLATE = "plotly_white"


def plot_equity_curve(result: BacktestResult) -> go.Figure:
    """Plot strategy equity versus benchmark equity."""

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=result.equity.index,
            y=result.equity,
            mode="lines",
            name=result.strategy_name,
            line=dict(width=2.5, color="#0F6B5B"),
        )
    )
    if result.benchmark_equity is not None:
        fig.add_trace(
            go.Scatter(
                x=result.benchmark_equity.index,
                y=result.benchmark_equity,
                mode="lines",
                name=result.config.benchmark,
                line=dict(width=1.8, color="#4C566A", dash="dot"),
            )
        )
    fig.update_layout(
        title="Equity Curve",
        template=CHART_TEMPLATE,
        yaxis_title="Portfolio Value",
        hovermode="x unified",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    )
    return fig


def plot_drawdown(result: BacktestResult) -> go.Figure:
    """Plot strategy and benchmark drawdowns."""

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=result.drawdown.index,
            y=result.drawdown,
            mode="lines",
            fill="tozeroy",
            name=f"{result.strategy_name} drawdown",
            line=dict(width=1.8, color="#B23A48"),
        )
    )
    if result.benchmark_equity is not None:
        benchmark_drawdown = drawdown_series(result.benchmark_equity)
        fig.add_trace(
            go.Scatter(
                x=benchmark_drawdown.index,
                y=benchmark_drawdown,
                mode="lines",
                name=f"{result.config.benchmark} drawdown",
                line=dict(width=1.4, color="#4C566A", dash="dot"),
            )
        )
    fig.update_layout(
        title="Drawdown",
        template=CHART_TEMPLATE,
        yaxis_title="Drawdown",
        yaxis_tickformat=".0%",
        hovermode="x unified",
    )
    return fig


def plot_rolling_sharpe(result: BacktestResult, window: int = 126) -> go.Figure:
    """Plot annualized rolling Sharpe ratio."""

    periods = 252
    returns = result.portfolio_returns
    rf_periodic = (1.0 + result.config.risk_free_rate) ** (1.0 / periods) - 1.0
    rolling = (returns - rf_periodic).rolling(window).mean() / returns.rolling(window).std()
    rolling = rolling * np.sqrt(periods)

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=rolling.index, y=rolling, mode="lines", name="Rolling Sharpe"))
    fig.add_hline(y=0, line_width=1, line_dash="dot", line_color="#4C566A")
    fig.update_layout(
        title=f"Rolling Sharpe Ratio ({window} sessions)",
        template=CHART_TEMPLATE,
        yaxis_title="Sharpe",
        hovermode="x unified",
    )
    return fig


def plot_weights(result: BacktestResult) -> go.Figure:
    """Plot realized portfolio weights through time."""

    fig = go.Figure()
    for column in result.weights.columns:
        fig.add_trace(
            go.Scatter(
                x=result.weights.index,
                y=result.weights[column],
                mode="lines",
                stackgroup="one",
                name=column,
            )
        )
    fig.update_layout(
        title="Realized Portfolio Weights",
        template=CHART_TEMPLATE,
        yaxis_title="Weight",
        yaxis_tickformat=".0%",
        hovermode="x unified",
    )
    return fig


def plot_monthly_returns(result: BacktestResult) -> go.Figure:
    """Plot monthly returns as a year by month heatmap."""

    monthly = (1.0 + result.portfolio_returns).resample("ME").prod() - 1.0
    heatmap = monthly.to_frame("return")
    heatmap["year"] = heatmap.index.year
    heatmap["month"] = heatmap.index.strftime("%b")
    pivot = heatmap.pivot(index="year", columns="month", values="return")
    month_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
    pivot = pivot.reindex(columns=month_order)

    fig = go.Figure(
        data=go.Heatmap(
            z=pivot.values,
            x=pivot.columns,
            y=pivot.index,
            colorscale="RdYlGn",
            zmid=0,
            colorbar=dict(title="Return"),
            hovertemplate="Year=%{y}<br>Month=%{x}<br>Return=%{z:.2%}<extra></extra>",
        )
    )
    fig.update_layout(title="Monthly Return Heatmap", template=CHART_TEMPLATE)
    return fig


def plot_metrics_table(result: BacktestResult) -> go.Figure:
    """Render a clean table of performance statistics."""

    formatted = result.metrics.copy()
    percent_rows = [
        "Total Return",
        "CAGR",
        "Annual Volatility",
        "Max Drawdown",
        "Best Period",
        "Worst Period",
        "Hit Rate",
        "VaR 95",
        "CVaR 95",
        "Benchmark CAGR",
        "Tracking Error",
        "Active Return",
    ]
    display_values = []
    for metric, value in formatted.items():
        if pd.isna(value):
            display_values.append("n/a")
        elif metric in percent_rows:
            display_values.append(f"{value:.2%}")
        else:
            display_values.append(f"{value:.2f}")

    fig = go.Figure(
        data=[
            go.Table(
                header=dict(values=["Metric", "Value"], fill_color="#0F6B5B", font=dict(color="white"), align="left"),
                cells=dict(values=[formatted.index, display_values], fill_color="#F7F9FB", align="left"),
            )
        ]
    )
    fig.update_layout(title="Performance Metrics", template=CHART_TEMPLATE, height=560)
    return fig


def build_tearsheet(result: BacktestResult) -> go.Figure:
    """Create a compact multi-panel interactive Plotly tear sheet."""

    fig = make_subplots(
        rows=3,
        cols=2,
        specs=[
            [{"colspan": 2}, None],
            [{"colspan": 2}, None],
            [{}, {}],
        ],
        subplot_titles=("Equity Curve", "Drawdown", "Rolling 126D Sharpe", "Turnover"),
        vertical_spacing=0.09,
    )
    fig.add_trace(go.Scatter(x=result.equity.index, y=result.equity, name=result.strategy_name), row=1, col=1)
    if result.benchmark_equity is not None:
        fig.add_trace(go.Scatter(x=result.benchmark_equity.index, y=result.benchmark_equity, name=result.config.benchmark), row=1, col=1)
    fig.add_trace(go.Scatter(x=result.drawdown.index, y=result.drawdown, name="Drawdown", fill="tozeroy"), row=2, col=1)

    rolling_sharpe = (
        result.portfolio_returns.rolling(126).mean()
        / result.portfolio_returns.rolling(126).std()
        * np.sqrt(252)
    )
    fig.add_trace(go.Scatter(x=rolling_sharpe.index, y=rolling_sharpe, name="Rolling Sharpe"), row=3, col=1)
    fig.add_trace(go.Bar(x=result.turnover.index, y=result.turnover, name="Turnover"), row=3, col=2)
    fig.update_layout(template=CHART_TEMPLATE, title="Portfolio Backtest Tear Sheet", height=900, hovermode="x unified")
    fig.update_yaxes(tickformat=".0%", row=2, col=1)
    return fig


## Colab Cell 12 - Optional vectorbt and backtrader Adapters


In [ ]:
%%writefile src/portfolio_backtester/adapters.py
"""Optional integrations with vectorbt and backtrader."""

from __future__ import annotations

import pandas as pd

from .config import BacktestConfig


def run_vectorbt_sma_strategy(prices: pd.DataFrame, config: BacktestConfig, ticker: str | None = None):
    """Run a simple vectorbt moving-average strategy for validation and comparison."""

    try:
        import vectorbt as vbt
    except ImportError as exc:
        raise RuntimeError("Install vectorbt to use this adapter: `pip install vectorbt`.") from exc

    symbol = (ticker or config.tickers[0]).upper()
    if symbol not in prices.columns:
        raise ValueError(f"{symbol} is not available in prices.")

    close = prices[symbol].dropna()
    fast_ma = vbt.MA.run(close, window=50)
    slow_ma = vbt.MA.run(close, window=200)
    entries = fast_ma.ma_crossed_above(slow_ma)
    exits = fast_ma.ma_crossed_below(slow_ma)
    return vbt.Portfolio.from_signals(
        close,
        entries=entries,
        exits=exits,
        init_cash=config.initial_cash,
        fees=config.commission_bps / 10_000.0,
        slippage=config.slippage_bps / 10_000.0,
        freq="1D",
    )


def run_backtrader_sma_strategy(prices: pd.DataFrame, config: BacktestConfig, ticker: str | None = None) -> float:
    """Run a minimal Backtrader SMA crossover and return final portfolio value."""

    try:
        import backtrader as bt
    except ImportError as exc:
        raise RuntimeError("Install backtrader to use this adapter: `pip install backtrader`.") from exc

    symbol = (ticker or config.tickers[0]).upper()
    if symbol not in prices.columns:
        raise ValueError(f"{symbol} is not available in prices.")

    class SmaCross(bt.Strategy):
        params = dict(fast=50, slow=200)

        def __init__(self):
            fast = bt.ind.SMA(period=self.p.fast)
            slow = bt.ind.SMA(period=self.p.slow)
            self.crossover = bt.ind.CrossOver(fast, slow)

        def next(self):
            if not self.position and self.crossover > 0:
                self.buy()
            elif self.position and self.crossover < 0:
                self.close()

    data = pd.DataFrame(
        {
            "open": prices[symbol],
            "high": prices[symbol],
            "low": prices[symbol],
            "close": prices[symbol],
            "volume": 0,
            "openinterest": 0,
        }
    ).dropna()

    cerebro = bt.Cerebro()
    cerebro.broker.setcash(config.initial_cash)
    cerebro.broker.setcommission(commission=config.commission_bps / 10_000.0)
    cerebro.addstrategy(SmaCross)
    cerebro.adddata(bt.feeds.PandasData(dataname=data))
    cerebro.addsizer(bt.sizers.PercentSizer, percents=95)
    cerebro.run()
    return float(cerebro.broker.getvalue())


## Colab Cell 13 - Dashboard Export Utilities


In [ ]:
%%writefile src/portfolio_backtester/dashboard.py
"""Export helpers for reports and tear sheets."""

from __future__ import annotations

from pathlib import Path

from .engine import BacktestResult
from .visualization import (
    build_tearsheet,
    plot_drawdown,
    plot_equity_curve,
    plot_metrics_table,
    plot_monthly_returns,
    plot_rolling_sharpe,
    plot_weights,
)


def export_html_report(result: BacktestResult, output_dir: str | Path = "reports") -> Path:
    """Write an interactive HTML tear sheet and return its path."""

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    report_file = output_path / f"{result.strategy_name}_tear_sheet.html"
    fig = build_tearsheet(result)
    fig.write_html(report_file, include_plotlyjs="cdn", full_html=True)
    return report_file


def build_dashboard_figures(result: BacktestResult):
    """Return all standard figures for notebook display."""

    return {
        "equity": plot_equity_curve(result),
        "drawdown": plot_drawdown(result),
        "rolling_sharpe": plot_rolling_sharpe(result),
        "weights": plot_weights(result),
        "monthly_returns": plot_monthly_returns(result),
        "metrics": plot_metrics_table(result),
        "tearsheet": build_tearsheet(result),
    }


## Colab Cell 14 - Import the Local Package

            Adds `src/` to the Python path and imports the framework.


In [ ]:

            import sys
            from pathlib import Path

            SRC_DIR = Path.cwd() / "src"
            if str(SRC_DIR.resolve()) not in sys.path:
                sys.path.insert(0, str(SRC_DIR.resolve()))

            import numpy as np
            import pandas as pd
            from IPython.display import display

            from portfolio_backtester import (
                BacktestConfig,
                BacktestEngine,
                DualMomentumStrategy,
                EqualWeightStrategy,
                VolatilityTargetMomentumStrategy,
                YahooFinanceDataLoader,
            )
            from portfolio_backtester.data import MarketData
            from portfolio_backtester.features import build_feature_panel
            from portfolio_backtester.dashboard import build_dashboard_figures, export_html_report

            print("Portfolio backtesting framework imported successfully.")


## Colab Cell 15 - Research Configuration

            Defines the investment universe, benchmark, date range, costs, leverage, and rebalancing assumptions.


In [ ]:

            config = BacktestConfig(
                tickers=("SPY", "QQQ", "TLT", "GLD"),
                benchmark="SPY",
                start="2015-01-01",
                end=None,
                initial_cash=100_000.0,
                commission_bps=1.0,
                slippage_bps=2.0,
                rebalance_frequency="M",
                risk_free_rate=0.02,
                max_gross_leverage=1.0,
                allow_short=False,
                min_observations=252,
            )

            config


## Colab Cell 16 - Data Download with Synthetic Fallback

            Downloads Yahoo Finance data. If the live API is unavailable during a demo, the fallback creates realistic synthetic prices so the full notebook still runs.


In [ ]:

            def make_synthetic_market_data(config: BacktestConfig, periods: int = 1_500, seed: int = 42) -> MarketData:
                rng = np.random.default_rng(seed)
                index = pd.bdate_range("2018-01-01", periods=periods)
                symbols = list(config.all_symbols)

                annual_drifts = np.linspace(0.05, 0.10, len(symbols))
                annual_vols = np.linspace(0.12, 0.22, len(symbols))
                daily_drifts = annual_drifts / 252
                daily_vols = annual_vols / np.sqrt(252)
                common_factor = rng.normal(0, 0.006, size=(periods, 1))
                idiosyncratic = rng.normal(daily_drifts, daily_vols, size=(periods, len(symbols)))
                simulated_returns = idiosyncratic + common_factor
                prices = pd.DataFrame(100 * np.exp(np.cumsum(simulated_returns, axis=0)), index=index, columns=symbols)
                returns = prices.pct_change(fill_method=None).fillna(0.0)
                volumes = pd.DataFrame(rng.integers(1_000_000, 8_000_000, size=prices.shape), index=index, columns=symbols)
                raw = prices.copy()
                return MarketData(prices=prices, returns=returns, volumes=volumes, raw=raw)


            loader = YahooFinanceDataLoader(config)

            try:
                market = loader.download()
                print("Downloaded live Yahoo Finance data.")
            except Exception as exc:
                print(f"Live Yahoo Finance download failed: {exc}")
                print("Using synthetic market data so the notebook remains executable.")
                market = make_synthetic_market_data(config)

            print(market.prices.index.min(), "to", market.prices.index.max())
            display(market.prices.tail())


## Colab Cell 17 - Feature Engineering

            Builds momentum, trend, volatility, RSI, and z-score features.


In [ ]:

            features = build_feature_panel(market.prices, market.returns)

            feature_preview = pd.concat(
                {
                    "momentum_126": features["momentum_126"].tail(3),
                    "volatility_63": features["volatility_63"].tail(3),
                    "rsi_14": features["rsi_14"].tail(3),
                },
                axis=1,
            )
            display(feature_preview)


## Colab Cell 18 - Run Strategy Backtests

            Runs equal weight, dual momentum, and volatility-targeted momentum through the same execution engine.


In [ ]:

            strategies = [
                EqualWeightStrategy(),
                DualMomentumStrategy(lookback=126, top_n=2),
                VolatilityTargetMomentumStrategy(momentum_lookback=126, volatility_lookback=63, trend_window=200),
            ]

            engine = BacktestEngine(config)
            results = {}

            for strategy in strategies:
                result = engine.run(market.prices, market.returns, strategy, features=features)
                results[strategy.name] = result
                print(f"{strategy.name}: final equity = ${result.equity.iloc[-1]:,.2f}")

            selected_name = "vol_target_momentum"
            result = results[selected_name]


## Colab Cell 19 - Performance Comparison Table

            Compares institutional metrics across all tested strategies.


In [ ]:

            comparison = pd.DataFrame({name: res.metrics for name, res in results.items()}).T

            percent_columns = [
                "Total Return",
                "CAGR",
                "Annual Volatility",
                "Max Drawdown",
                "Hit Rate",
                "VaR 95",
                "CVaR 95",
                "Benchmark CAGR",
                "Tracking Error",
                "Active Return",
            ]

            formatted = comparison.copy()
            for column in formatted.columns:
                if column in percent_columns:
                    formatted[column] = formatted[column].map(lambda value: f"{value:.2%}" if pd.notna(value) else "n/a")
                else:
                    formatted[column] = formatted[column].map(lambda value: f"{value:.2f}" if pd.notna(value) else "n/a")

            display(formatted)


## Colab Cell 20 - Equity Curve

            Plots the selected strategy against the benchmark.


In [ ]:

            figures = build_dashboard_figures(result)
            figures["equity"].show()


## Colab Cell 21 - Drawdown Chart

            Visualizes peak-to-trough losses, one of the most important professional risk diagnostics.


In [ ]:
figures["drawdown"].show()


## Colab Cell 22 - Rolling Sharpe Ratio

            Checks whether risk-adjusted performance is stable or concentrated in a few periods.


In [ ]:
figures["rolling_sharpe"].show()


## Colab Cell 23 - Portfolio Weights

            Shows realized allocations through time, including cash when weights sum below 100 percent.


In [ ]:
figures["weights"].show()


## Colab Cell 24 - Monthly Return Heatmap

            Gives a regime-level view of returns by year and month.


In [ ]:
figures["monthly_returns"].show()


## Colab Cell 25 - Metrics Table

            Renders a compact professional performance table.


In [ ]:
figures["metrics"].show()


## Colab Cell 26 - Export Research Artifacts

            Saves metrics, equity curves, weights, turnover, costs, and an interactive HTML tear sheet.


In [ ]:

            reports_dir = Path("reports")
            reports_dir.mkdir(exist_ok=True)

            comparison.to_csv(reports_dir / "strategy_comparison.csv")
            result.metrics.to_csv(reports_dir / f"{result.strategy_name}_metrics.csv", header=["value"])
            result.equity.to_csv(reports_dir / f"{result.strategy_name}_equity.csv", header=["equity"])
            result.weights.to_csv(reports_dir / f"{result.strategy_name}_weights.csv")
            result.turnover.to_csv(reports_dir / f"{result.strategy_name}_turnover.csv", header=["turnover"])
            result.costs.to_csv(reports_dir / f"{result.strategy_name}_costs.csv", header=["costs"])

            report_path = export_html_report(result, reports_dir)
            print(f"Saved report: {report_path}")


## Colab Cell 27 - Optional vectorbt Validation

            Runs a simple single-asset SMA crossover in vectorbt if the optional dependency installed successfully.


In [ ]:

            try:
                from portfolio_backtester.adapters import run_vectorbt_sma_strategy

                vectorbt_portfolio = run_vectorbt_sma_strategy(market.prices, config, ticker="SPY")
                print(vectorbt_portfolio.stats())
            except Exception as exc:
                print(f"vectorbt validation skipped: {exc}")


## Colab Cell 28 - Optional Backtrader Validation

            Runs the same style of simple SMA crossover in Backtrader if available.


In [ ]:

            try:
                from portfolio_backtester.adapters import run_backtrader_sma_strategy

                final_value = run_backtrader_sma_strategy(market.prices, config, ticker="SPY")
                print(f"Backtrader SMA final value: ${final_value:,.2f}")
            except Exception as exc:
                print(f"Backtrader validation skipped: {exc}")


## Colab Cell 29 - Build Your Own Strategy

            To extend the framework, subclass `BaseStrategy` and return a DataFrame of target weights indexed like `prices`. The engine handles execution timing, transaction costs, constraints, analytics, and reporting.


In [ ]:

            from portfolio_backtester.strategies import BaseStrategy, rebalance_mask, normalize_long_only


            class RsiMeanReversionStrategy(BaseStrategy):
                name = "rsi_mean_reversion"

                def generate_weights(self, prices, returns, config, features=None):
                    universe = list(config.tickers)
                    rsi_values = (features or build_feature_panel(prices, returns))["rsi_14"][universe]
                    scores = (50 - rsi_values).clip(lower=0)
                    weights = normalize_long_only(scores, config.max_gross_leverage)
                    mask = rebalance_mask(prices.index, config.rebalance_frequency)
                    return weights.where(mask, np.nan).ffill().fillna(0.0)


            custom_result = engine.run(market.prices, market.returns, RsiMeanReversionStrategy(), features=features)
            print(custom_result.metrics[["CAGR", "Sharpe Ratio", "Max Drawdown"]])
